<a href="https://colab.research.google.com/github/pedrodiniz474/Notebook-Atividades-Fabiano-I.A-6C-Noturno-PedroDiniz-/blob/main/Atividade_Machine_Learning_Brasileirao_%5BPedroDiniz%5D_%5B6C%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tema: Previsão se o time da casa ganha a partida com Machine Learning

Dataset utilizado: https://www.kaggle.com/datasets/adaoduque/campeonato-brasileiro-de-futebol (Só baixar e extrair o CSV específico na aba arquivos do lado esquerdo do colab)

CSV utilizado: campeonato-brasileiro-full.csv

Integrantes

Nome: MATHEUS BARBOSA NUNES DE OLIVEIRA RGM: 11261405334

Nome: PEDRO HENRIQUE DINIZ MATOS RGM: 11241103789

1. BIBLIOTECAS

In [1]:
#pandas para mexer na tabela
import pandas as pd


#sklearn para separar os dados, normalizar, treinar o SVM e testar
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

 Pandas lê o arquivo e organiza a tabela. O sklearn é a biblioteca de machine learning.

2. CARREGAMENTO DOS DADOS

In [2]:
#lê o arquivo CSV que está salvo no Colab
dados = pd.read_csv('campeonato-brasileiro-full.csv')

#mostra as primeiras linhas
dados.head()

,ID,rodata,data,hora,mandante,visitante,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,vencedor,arena,mandante_Placar,visitante_Placar,mandante_Estado,visitante_Estado,arrecadacao
0,1,1,29/03/2003,16:00,Guarani,Vasco,NaN,NaN,NaN,NaN,Guarani,Brinco de Ouro,4,2,SP,RJ,NaN
1,2,1,29/03/2003,16:00,Athletico-PR,Gremio,NaN,NaN,NaN,NaN,Athletico-PR,Arena da Baixada,2,0,PR,RS,NaN
2,3,1,30/03/2003,16:00,Flamengo,Coritiba,NaN,NaN,NaN,NaN,-,Maracanã,1,1,RJ,PR,NaN
3,4,1,30/03/2003,16:00,Goias,Paysandu,NaN,NaN,NaN,NaN,-,Serra Dourada,2,2,GO,PA,NaN
4,5,1,30/03/2003,16:00,Internacional,Ponte Preta,NaN,NaN,NaN,NaN,-,Beira Rio,1,1,RS,SP,NaN


A gente carrega o arquivo campeonato-brasileiro-full.csv. O comando head() serve para ver as primeiras linhas e entender quais colunas existem, como o placar, os times, a rodada e as formações táticas.

3. LIMPEZA DOS DADOS

In [3]:
#tira as linhas com valores vazios e as duplicadas
dados = dados.dropna()
dados = dados.drop_duplicates()

#cria a coluna que queremos prever: 1 se o mandante ganhou, 0 se não
dados['resultado'] = (dados['mandante_Placar'] > dados['visitante_Placar']).astype(int)


O dropna() remove as linhas com valores vazios e o drop_duplicates() tira as linhas repetidas. Depois, criamos a coluna resultado, que é o nosso alvo. Se o time da casa fez mais gols, vale 1. Se não, vale 0.

4. ENCODE E NORMALIZAÇÃO

In [5]:
#escolhe as colunas de entrada
colunas = ['rodata', 'mandante_Estado', 'formacao_mandante']
X = dados[colunas]
y = dados['resultado']

#encode: transforma os textos em números (0 e 1)
X = pd.get_dummies(X, columns=['mandante_Estado', 'formacao_mandante'])

#normaliza: coloca tudo na mesma escala
scaler = StandardScaler()
X = scaler.fit_transform(X)

O SVM não trabalha com texto. Então, usamos o pd.get_dummies para fazer o Encode, transformando os nomes dos estados e das formações em colunas de 0 e 1.

5. DIVISÃO DOS DADOS

In [6]:
#separa 80% para treinar e 20% para testar o modelo
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

O train_test_split separa uma parte para o treino e guarda a outra parte para a prova final.

6. APLICAR SVC E GRINDSEARCHCV

In [7]:
#define os parâmetros que o GridSearch vai testar
parametros = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}

#cria o modelo SVC com o GridSearchCV e treina
grid = GridSearchCV(SVC(), parametros, cv=3)
grid.fit(X_treino, y_treino)

GridSearchCV(cv=3, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']})

Usamos o SVC para classificação. E usamos o GridSearchCV para testar várias combinações de C e kernel e ver qual funciona melhor.

7. AVALIAÇÃO DO MODELO

In [8]:
#faz as previsões usando o melhor modelo encontrado
previsoes = grid.predict(X_teste)

#calcula a acurácia
acuracia = accuracy_score(y_teste, previsoes)
print(f'Acurácia: {acuracia * 100:.2f}%')
print(f'Melhores parâmetros: {grid.best_params_}')

Acurácia: 44.74%
Melhores parâmetros: {'C': 1, 'kernel': 'rbf'}


Depois de treinar, a gente usa o modelo para prever os resultados dos dados de teste. A acurácia mostra quantos por cento o modelo acertou. Também imprimimos o best_params_ para ver qual configuração de SVM o GridSearch escolheu como a melhor. Se a acurácia for boa, significa que o modelo conseguiu aprender os padrões do campeonato.